# สกัดจุดดึงดูดประชากรเชิงพื้นที่ (Demand POIs Extraction)
สมุดโน้ตเล่มนี้ถูกปรับปรุงให้ดึงข้อมูลพิกัดอาคารสำนักงาน คอนโด หอพัก และห้างสรรพสินค้า ผ่าน **Overpass API** (ดึงผ่านอินเทอร์เน็ตโดยตรง) ทำให้ไม่ต้องโหลดและแกะไฟล์แผนที่ดิบ `osm.pbf` ขนาด 324MB ในเครื่อง ซึ่งจะช่วยย่นระยะเวลาการประมวลผลให้รวดเร็วและไม่กินแรมของคอมพิวเตอร์

In [ ]:
import pandas as pd
import requests
import os
import gc
from pathlib import Path

# ค้นหาตำแหน่งโฟลเดอร์หลักของโปรเจกต์
NOTEBOOK_DIR = Path(os.getcwd())
BASE_DIR = NOTEBOOK_DIR.parent.parent # เลื่อนขึ้นไปที่ ZoneVision/data-pipeline

print(f"Notebook Directory: {NOTEBOOK_DIR}")
print(f"Base Directory: {BASE_DIR}")

In [ ]:
# โหลดตั้งค่าขอบเขตรอยต่อกรุงเทพฯ (BBox) จากคอนฟิก
import json
config_path = BASE_DIR / "config.json"
with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

bkk_bbox = config.get("bkk_bbox", [100.30, 13.45, 100.95, 13.95])
target_pois = config.get("target_pois", ["apartments", "residential", "office", "commercial", "retail"])

# Bbox format for Overpass API
bbox_str = f"{bkk_bbox[1]},{bkk_bbox[0]},{bkk_bbox[3]},{bkk_bbox[2]}"
print(f"Bangkok BBox: {bbox_str}")
print(f"Target POIs/Buildings: {target_pois}")

In [ ]:
# 1. ดึงข้อมูลตึกจาก Overpass API
print("กำลังดาวน์โหลดข้อมูลตึกประชากรจาก Overpass API (ขั้นตอนนี้ขนาดยอดตึกมีมาก อาจใช้เวลา 10-30 วินาที)...")
overpass_url = "http://overpass-api.de/api/interpreter"
building_regex = "|".join(target_pois)

query_str = f"""
[out:json][timeout:180];
(
  node["building"~"{building_regex}"]({bbox_str});
  way["building"~"{building_regex}"]({bbox_str});
  relation["building"~"{building_regex}"]({bbox_str});
);
out center;
"""

headers = {
    'User-Agent': 'ZoneVisionSeniorProject/1.0 (contact: naeiger@example.com)'
}

try:
    response = requests.post(overpass_url, data={'data': query_str}, headers=headers, timeout=180)
    if response.status_code == 200:
        data = response.json()
        elements = data.get('elements', [])
        print(f"🎉 ดึงข้อมูลสำเร็จ! พบรายการสิ่งปลูกสร้างในระบบ: {len(elements)} อาคาร")
        
        pois = []
        for el in elements:
            lat = el.get('lat') or el.get('center', {}).get('lat')
            lon = el.get('lon') or el.get('center', {}).get('lon')
            tags = el.get('tags', {})
            name = tags.get('name') or tags.get('name:en') or tags.get('name:th')
            building = tags.get('building')
            
            pois.append({
                'name': name,
                'building': building,
                'latitude': lat,
                'longitude': lon
            })
            
        df_flat = pd.DataFrame(pois)
        print(df_flat.head(10))
    else:
        print(f"❌ การดึงข้อมูลล้มเหลว: HTTP Code {response.status_code}")
except Exception as e:
    print(f"⚠️ เกิดข้อผิดพลาดในการดึงข้อมูล: {str(e)}")

In [ ]:
# 2. บันทึกผลลัพธ์ข้อมูลระดับดิบลงในคลัง interim
if 'df_flat' in locals() and len(df_flat) > 0:
    os.makedirs(str(BASE_DIR / "data" / "interim"), exist_ok=True)
    output_file = str(BASE_DIR / "data" / "interim" / "bangkok_pois.json")
    df_flat.to_json(output_file, orient='records', force_ascii=False, indent=4)
    print(f"💾 บันทึกไฟล์ระดับดิบเข้าคลัง interim สำเร็จ: {output_file}")
    
    # เคลียร์แรม
    del df_flat
    gc.collect()